# Banco de Dados de Análise de Crédito

## Objetivo
 
Construir um modelo de classificação binária para prever o **risco de crédito** de clientes,
com base em variáveis financeiras, pessoais e comportamentais.
 
| | |
|---|---|
| **Variável Alvo** | `risco_credito` |
| **Tipo** | Binária: `0` = bom pagador, `1` = mau pagador |
| **Tarefa** | Classificação supervisionada |
 
---
 
## Fonte dos Dados
 
- Dataset: German Credit Data
- Arquivo: `german_credit_data.csv`
- Armazenamento: banco SQLite (`analise_credito_alemao.db`), tabela `credito_clientes`
---
 
## Variáveis do Dataset
 
### Numéricas
| Coluna | Descrição |
|---|---|
| `duracao_meses` | Duração do crédito em meses |
| `valor_credito` | Valor do crédito solicitado |
| `idade` | Idade do cliente |
 
### Categóricas Ordinais
| Coluna | Descrição |
|---|---|
| `status_conta_corrente` | Situação da conta corrente |
| `poupanca_investimento` | Nível de poupança/investimento |
| `tempo_emprego_atual` | Tempo no emprego atual |
| `taxa_parcelamento_renda` | Taxa de parcelamento em relação à renda |
| `residencia_atual_desde` | Tempo na residência atual |
| `numero_creditos_existentes` | Quantidade de créditos ativos |
| `dependentes` | Número de pessoas dependentes |
 
### Categóricas Nominais
| Coluna | Descrição |
|---|---|
| `historico_credito` | Histórico de pagamentos |
| `proposito` | Finalidade do crédito |
| `status_pessoal_sexo` | Estado civil e sexo |
| `outros_fiadores` | Existência de fiadores |
| `propriedade` | Tipo de propriedade |
| `outros_planos_parcelamento` | Outros planos de parcelamento |
| `habitacao` | Tipo de habitação |
| `trabalho` | Cargo/ocupação |
| `telefone` | Possui telefone |
| `trabalhador_estrangeiro` | É trabalhador estrangeiro |
 
---

In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from importlib import reload
import sqlite3 # Usar SQLite para facilitar o upload de um arquivo único ao GitHub
import seaborn as sns

In [2]:
# Carregar dados
df = pd.read_csv('data/german_credit_data.csv')

# Dicionário de tradução (ajuste conforme sua interpretação das variáveis)
colunas_pt = {
    'laufkont': 'status_conta_corrente',
    'laufzeit': 'duracao_meses',
    'moral': 'historico_credito',
    'verw': 'proposito',
    'hoehe': 'valor_credito',
    'sparkont': 'poupanca_investimento',
    'beszeit': 'tempo_emprego_atual',
    'rate': 'taxa_parcelamento_renda',
    'famges': 'status_pessoal_sexo',
    'buerge': 'outros_fiadores',
    'wohnzeit': 'residencia_atual_desde',
    'verm': 'propriedade',
    'alter': 'idade',
    'weitkred': 'outros_planos_parcelamento',
    'wohn': 'habitacao',
    'bishkred': 'numero_creditos_existentes',
    'beruf': 'trabalho',
    'pers': 'dependentes',
    'telef': 'telefone',
    'gastarb': 'trabalhador_estrangeiro',
    'kredit': 'risco_credito'
}

df.rename(columns=colunas_pt, inplace=True)

# Criando o banco de dados local (gerando um arquivo .db)
conn = sqlite3.connect('analise_credito_alemao.db')
df.to_sql('credito_cliente', conn, if_exists='replace', index=False)

1000

In [3]:
def run_query(query):
    return pd.read_sql_query(query, conn)

# Limpeza de Dados

In [4]:
query = """
SELECT
    -- ========================
    -- VARIÁVEIS NUMÉRICAS
    -- ========================

    CASE 
        WHEN duracao_meses <= 0 THEN NULL
        ELSE duracao_meses 
    END AS duracao_meses,

    CASE 
        WHEN valor_credito <= 0 THEN NULL
        ELSE valor_credito 
    END AS valor_credito,

    CASE 
        WHEN idade < 18 OR idade > 100 THEN NULL
        ELSE idade 
    END AS idade,

    -- ========================
    -- VARIÁVEL ALVO
    -- ========================

    CASE 
        WHEN risco_credito NOT IN (0,1) THEN NULL
        ELSE risco_credito 
    END AS risco_credito,

    
    -- ========================
    -- VARIÁVEL CATEGÓRICAS (com cast para INTEGER)
    -- ========================

     CASE
        WHEN CAST(status_conta_corrente AS INTEGER) BETWEEN 0 AND 4
        THEN CAST(status_conta_corrente AS INTEGER)
        ELSE NULL
    END AS status_conta_corrente,

    CASE
        WHEN CAST(historico_credito AS INTEGER) BETWEEN 0 AND 4
        THEN CAST(historico_credito AS INTEGER)
        ELSE NULL
    END AS historico_credito,

    CASE
        WHEN CAST(proposito AS INTEGER) BETWEEN 0 AND 10
        THEN CAST(proposito AS INTEGER)
        ELSE NULL
    END AS proposito,

    CASE
        WHEN CAST(poupanca_investimento AS INTEGER) BETWEEN 0 AND 6
        THEN CAST(poupanca_investimento AS INTEGER)
        ELSE NULL
    END AS poupanca_investimento,

    CASE
        WHEN CAST(tempo_emprego_atual AS INTEGER) BETWEEN 0 AND 5
        THEN CAST(tempo_emprego_atual AS INTEGER)
        ELSE NULL
    END AS tempo_emprego_atual,

    CASE
        WHEN CAST(taxa_parcelamento_renda AS INTEGER) BETWEEN 0 AND 4
        THEN CAST(taxa_parcelamento_renda AS INTEGER)
        ELSE NULL
    END AS taxa_parcelamento_renda,

    CASE
        WHEN CAST(status_pessoal_sexo AS INTEGER) BETWEEN 0 AND 4
        THEN CAST(status_pessoal_sexo AS INTEGER)
        ELSE NULL
    END AS status_pessoal_sexo,

    CASE
        WHEN CAST(outros_fiadores AS INTEGER) BETWEEN 0 AND 3
        THEN CAST(outros_fiadores AS INTEGER)
        ELSE NULL
    END AS outros_fiadores,

    CASE
        WHEN CAST(residencia_atual_desde AS INTEGER) BETWEEN 0 AND 4
        THEN CAST(residencia_atual_desde AS INTEGER)
        ELSE NULL
    END AS residencia_atual_desde,

    CASE
        WHEN CAST(propriedade AS INTEGER) BETWEEN 0 AND 4
        THEN CAST(propriedade AS INTEGER)
        ELSE NULL
    END AS propriedade,

    CASE
        WHEN CAST(outros_planos_parcelamento AS INTEGER) BETWEEN 0 AND 3
        THEN CAST(outros_planos_parcelamento AS INTEGER)
        ELSE NULL
    END AS outros_planos_parcelamento,

    CASE
        WHEN CAST(habitacao AS INTEGER) BETWEEN 0 AND 3
        THEN CAST(habitacao AS INTEGER)
        ELSE NULL
    END AS habitacao,

    CASE
        WHEN CAST(numero_creditos_existentes AS INTEGER) BETWEEN 0 AND 4
        THEN CAST(numero_creditos_existentes AS INTEGER)
        ELSE NULL
    END AS numero_creditos_existentes,

    CASE
        WHEN CAST(trabalho AS INTEGER) BETWEEN 0 AND 4
        THEN CAST(trabalho AS INTEGER)
        ELSE NULL
    END AS trabalho,

    CASE
        WHEN CAST(dependentes AS INTEGER) BETWEEN 0 AND 2
        THEN CAST(dependentes AS INTEGER)
        ELSE NULL
    END AS dependentes,

    CASE
        WHEN CAST(telefone AS INTEGER) BETWEEN 0 AND 2
        THEN CAST(telefone AS INTEGER)
        ELSE NULL
    END AS telefone,

    CASE
        WHEN CAST(trabalhador_estrangeiro AS INTEGER) BETWEEN 0 AND 2
        THEN CAST(trabalhador_estrangeiro AS INTEGER)
        ELSE NULL
    END AS trabalhador_estrangeiro
    

FROM credito_cliente

-- Foco apenas no target
WHERE risco_credito IN (0,1)
"""

df_limpo = run_query(query)

print(f"\nNulos por coluna:\n{df_limpo.isnull().sum()}")
print(f"\nValores Duplicados:\n{df_limpo.duplicated().sum()}")

# Substitui valores nulos pela mediana da coluna
df_limpo.fillna(df.median(), inplace=True)
df_limpo = df_limpo.drop_duplicates()

print(f"Linhas originais  : {len(run_query('SELECT * FROM credito_cliente'))}")
print(f"Linhas após limpeza: {len(df_limpo)}")



Nulos por coluna:
duracao_meses                 0
valor_credito                 0
idade                         0
risco_credito                 0
status_conta_corrente         0
historico_credito             0
proposito                     0
poupanca_investimento         0
tempo_emprego_atual           0
taxa_parcelamento_renda       0
status_pessoal_sexo           0
outros_fiadores               0
residencia_atual_desde        0
propriedade                   0
outros_planos_parcelamento    0
habitacao                     0
numero_creditos_existentes    0
trabalho                      0
dependentes                   0
telefone                      0
trabalhador_estrangeiro       0
dtype: int64

Valores Duplicados:
0
Linhas originais  : 1000
Linhas após limpeza: 1000


In [7]:
# Verificar se o dataset está balanceado
df["risco_credito"].value_counts()

risco_credito
1    700
0    300
Name: count, dtype: int64

In [5]:
df_limpo.head()

,duracao_meses,valor_credito,idade,risco_credito,status_conta_corrente,historico_credito,proposito,poupanca_investimento,tempo_emprego_atual,taxa_parcelamento_renda,...,outros_fiadores,residencia_atual_desde,propriedade,outros_planos_parcelamento,habitacao,numero_creditos_existentes,trabalho,dependentes,telefone,trabalhador_estrangeiro
0,18,1049,21,1,1,4,2,1,2,4,...,1,4,2,3,1,1,3,2,1,2
1,9,2799,36,1,1,4,0,1,3,2,...,1,2,1,3,1,2,3,1,1,2
2,12,841,23,1,2,2,9,2,4,2,...,1,4,1,3,1,1,2,2,1,2
3,12,2122,39,1,1,4,0,1,3,3,...,1,2,1,3,1,2,2,1,1,1
4,12,2171,38,1,1,4,0,1,3,4,...,1,4,2,1,2,2,2,2,1,1


In [6]:
df_limpo.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   duracao_meses               1000 non-null   int64
 1   valor_credito               1000 non-null   int64
 2   idade                       1000 non-null   int64
 3   risco_credito               1000 non-null   int64
 4   status_conta_corrente       1000 non-null   int64
 5   historico_credito           1000 non-null   int64
 6   proposito                   1000 non-null   int64
 7   poupanca_investimento       1000 non-null   int64
 8   tempo_emprego_atual         1000 non-null   int64
 9   taxa_parcelamento_renda     1000 non-null   int64
 10  status_pessoal_sexo         1000 non-null   int64
 11  outros_fiadores             1000 non-null   int64
 12  residencia_atual_desde      1000 non-null   int64
 13  propriedade                 1000 non-null   int64
 14  outros_planos_parcel

# EDA

## Análise Univariada

In [ ]:
colunas = df.select_dtypes(include=[np.number]).columns
resultados = []

for coluna in colunas:
    serie = df[coluna]

    estatisticas = {
        "coluna": coluna,
        
        # Tendência central
        "media": serie.mean(),
        "mediana": serie.median(),
        "moda": serie.mode().iloc[0] if not serie.mode().empty else np.nan,

        # Dispersão
        "variancia": serie.var(),
        "desvio_padrao": serie.std(),
        "coeficiente_variacao": (
            serie.std() / serie.mean()
            if serie.mean() != 0 else np.nan
        ),

        # Extremos
        "minimo": serie.min(),
        "maximo": serie.max(),
        "amplitude": serie.max() - serie.min(),

        # Quartis
        "q1": serie.quantile(0.25),
        "q2_mediana": serie.quantile(0.50),
        "q3": serie.quantile(0.75),
        "iqr": serie.quantile(0.75) - serie.quantile(0.25),

        # Distribuição
        "assimetria": serie.skew(),
        "curtose": serie.kurt(),

        # Percentis adicionais
        "p5": serie.quantile(0.05),
        "p95": serie.quantile(0.95),

        # Outliers pelo método IQR
        "outliers_iqr": (
            ((serie < (serie.quantile(0.25) - 1.5 * (
                serie.quantile(0.75) - serie.quantile(0.25)))) |
                (serie > (serie.quantile(0.75) + 1.5 * (
                serie.quantile(0.75) - serie.quantile(0.25))))).sum()
        )
    }

    resultados.append(estatisticas)

# Cria dataframe estatisticas
df_estatisticas = pd.DataFrame(resultados)
df_estatisticas


,coluna,media,mediana,moda,variancia,desvio_padrao,coeficiente_variacao,minimo,maximo,amplitude,q1,q2_mediana,q3,iqr,assimetria,curtose,p5,p95,outliers_iqr
0,status_conta_corrente,2.577,2.0,4,1.581653e+00,1.257638,0.488024,1,4,3,1.0,2.0,4.00,3.00,0.006957,-1.663703,1.00,4.0,0
1,duracao_meses,20.903,18.0,24,1.454150e+02,12.058814,0.576894,4,72,68,12.0,18.0,24.00,12.00,1.094184,0.919781,6.00,48.0,70
2,historico_credito,2.545,2.0,2,1.173148e+00,1.083120,0.425587,0,4,4,2.0,2.0,4.00,2.00,-0.011886,-0.579056,1.00,4.0,0
3,proposito,2.828,2.0,3,7.531948e+00,2.744439,0.970452,0,10,10,1.0,2.0,3.00,2.00,1.178887,0.554083,0.00,9.0,118
4,valor_credito,3271.248,2319.5,1258,7.967927e+06,2822.751760,0.862898,250,18424,18174,1365.5,2319.5,3972.25,2606.75,1.949594,4.292481,708.95,9162.7,72
5,poupanca_investimento,2.105,1.0,1,2.496471e+00,1.580023,0.750605,1,5,4,1.0,1.0,3.00,2.00,1.016677,-0.680224,1.00,5.0,0
6,tempo_emprego_atual,3.384,3.0,3,1.460004e+00,1.208306,0.357064,1,5,4,3.0,3.0,5.00,2.00,-0.117615,-0.934331,1.00,5.0,0
7,taxa_parcelamento_renda,2.973,3.0,4,1.251523e+00,1.118715,0.376292,1,4,3,2.0,3.0,4.00,2.00,-0.531348,-1.210473,1.00,4.0,0
8,status_pessoal_sexo,2.682,3.0,3,5.013774e-01,0.708080,0.264012,1,4,3,2.0,3.0,3.00,1.00,-0.305146,-0.002567,1.95,4.0,0
9,outros_fiadores,1.145,1.0,1,2.282032e-01,0.477706,0.417211,1,3,2,1.0,1.0,1.00,0.00,3.264249,9.328756,1.00,3.0,93


# Resumo dos Ajustes - Análise de Crédito Alemão

## 1. Carregamento dos Dados

- Dados carregados via `pd.read_csv` a partir da pasta `data/`
- Caminho correto: `data/german_credit_data.csv` (relativo ao notebook)
- Colunas renomeadas do alemão para português via dicionário `colunas_pt`

---

## 2. Banco de Dados

- Banco criado com SQLite via `sqlite3.connect('analise_credito_alemao.db')`
- Tabela criada automaticamente com `df.to_sql` — arquivo SQL separado descartado por ser redundante
- Nome padronizado da tabela: `credito_clientes` 

---

## 3. Limpeza dos Dados via SQL

### Variáveis Numéricas
| Coluna | Tratativa | Motivo |
|---|---|---|
| `duracao_meses` | Negativo ou zero vira `NULL` | Duração não pode ser inválida |
| `valor_credito` | Negativo ou zero vira `NULL` | Valor não pode ser inválido |
| `idade` | Fora de 18–100 vira `NULL` | Idades impossíveis |

### Variável Alvo
| Coluna | Tratativa | Motivo |
|---|---|---|
| `risco_credito` | Apenas `0` ou `1` aceitos | Variável binária, não pode ter outro valor |

### Categóricas
- Todas as colunas categóricas são convertidas para INTEGER (CAST).
- Validação de domínio (faixas esperadas): cada coluna é verificada com BETWEEN; valores fora da faixa viram NULL.

### Cláusula WHERE
- Foco apenas no target: `WHERE risco_credito IN (0,1)`
- Demais tratativas ficam no `CASE` — transformam valores inválidos em `NULL` sem remover a linha

---

## 4. Resultados da Limpeza
- Exibe a contagem de valores nulos por coluna antes da remoção.
- Exibe a quantidade de linhas duplicadas.
- Mostra quantas linhas existiam originalmente na tabela SQLite e quantas restaram após dropna() + drop_duplicates().